# 문단 기반 청킹 테스트

실제 구현은 같은 폴더의 `paragraph_chunk.py`에 있습니다.
이 노트북은 함수를 불러와 전처리·청킹 결과만 확인하며, API·DB를 사용하지 않습니다.


In [ ]:
# 저장소 루트 또는 src/rag_home_api/test/bc에서 실행할 수 있도록 모듈 위치를 찾는다.
from pathlib import Path
import importlib
import sys

import pandas as pd

module_dir = Path.cwd()
if not (module_dir / "paragraph_chunk.py").is_file():
    module_dir = module_dir / "src/rag_home_api/test/bc"
if not (module_dir / "paragraph_chunk.py").is_file():
    raise FileNotFoundError("저장소 루트 또는 src/rag_home_api/test/bc에서 실행해 주세요.")
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

# .py를 수정한 뒤 이 셀을 재실행하면 최신 함수를 다시 읽는다.
import paragraph_chunk as chunk_module
chunk_module = importlib.reload(chunk_module)
csv_path = chunk_module.DEFAULT_CSV_PATH


## 1. 전처리·Document 생성 확인

원본 CSV를 읽고 제외 컬럼·자료형·생성 개수를 확인합니다.


In [ ]:
# Python 파일의 전처리 함수를 사용한다. 노트북에 구현을 중복 작성하지 않는다.
df = chunk_module.load_and_clean(csv_path)
documents = chunk_module.create_documents(df, csv_path)

# 전처리에서 제외한 항목과 거래별 Document 생성을 확인한다.
assert {"prev_deposit_manwon", "prev_rent_manwon", "renewal_right_used"}.isdisjoint(df.columns)
assert len(documents) == len(df)
print("정제 거래 수:", len(df))
print("Document 수:", len(documents))
display(df[["id", "region_dong", "area_m2", "deposit_manwon", "monthly_rent_manwon"]].head(3))


## 2. 문단 기반 청킹 검증

청크 ID 중복·원본 거래 누락·빈 본문이 없는지 검사합니다.


In [ ]:
# 청킹 함수는 .py에 있고, 이 셀은 결과를 검사한다.
chunks = chunk_module.paragraph_chunk(documents)

# 원본 거래 식별자는 metadata.id, 청크 식별자는 Document.id다.
assert len({chunk.id for chunk in chunks}) == len(chunks)
assert {chunk.metadata["id"] for chunk in chunks} == {doc.metadata["id"] for doc in documents}
assert all(chunk.page_content.strip() for chunk in chunks)

# 길이 분포와 개수를 표로 확인한다.
lengths = pd.Series([len(chunk.page_content) for chunk in chunks], dtype="int64")
display(pd.DataFrame([{
    "원본 거래 수": len(documents), "청크 수": len(chunks),
    "평균 길이": round(lengths.mean(), 2), "최소 길이": lengths.min(),
    "중앙값 길이": lengths.median(), "최대 길이": lengths.max(),
}]))


## 3. 첫 청크 내용 확인


In [ ]:
# 전체 결과를 길게 출력하지 않고 첫 번째 청크만 확인한다.
if chunks:
    print("청크 ID:", chunks[0].id)
    print("\n[본문]")
    print(chunks[0].page_content)
    print("\n[메타데이터]")
    print(chunks[0].metadata)
